In [1]:
import gymnasium as gym
from stable_baselines3 import DQN as SB3_DQN
from stable_baselines3.common.evaluation import evaluate_policy

env = gym.make("LunarLander-v3")

episodes = 100

for ep in range(episodes):
    
    # Reset environment to start a new episode
    observation, info = env.reset()
    total_reward = 0
    episode_over = False
    print(f"Episode number : {ep}")

    while not episode_over:
        action = 0

        observation, reward, terminated, truncated, info = env.step(action)
        
        total_reward += reward
        episode_over = terminated or truncated

    print(f"Episode finished! Total reward: {total_reward}")

env.close()

Episode number : 0
Episode finished! Total reward: -156.1606706760784
Episode number : 1
Episode finished! Total reward: -100.17922532395039
Episode number : 2
Episode finished! Total reward: -115.70997991362225
Episode number : 3
Episode finished! Total reward: -116.45899719359122
Episode number : 4
Episode finished! Total reward: -136.47684647424535
Episode number : 5
Episode finished! Total reward: -118.22453215091872
Episode number : 6
Episode finished! Total reward: -123.0550556311997
Episode number : 7
Episode finished! Total reward: -140.58569990163633
Episode number : 8
Episode finished! Total reward: -149.28107112052155
Episode number : 9
Episode finished! Total reward: -112.77065699388211
Episode number : 10
Episode finished! Total reward: -138.915274428489
Episode number : 11
Episode finished! Total reward: -120.83544819618717
Episode number : 12
Episode finished! Total reward: -157.22639988567738
Episode number : 13
Episode finished! Total reward: -103.80417974766303
Episod

In [2]:
import gymnasium as gym
from stable_baselines3 import DQN as SB3_DQN
from stable_baselines3.common.evaluation import evaluate_policy

# ------------------------------------------------------------------
# 1. Environnement
# ------------------------------------------------------------------
env = gym.make("LunarLander-v3")

# ------------------------------------------------------------------
# 2. Instanciation du modèle
# ------------------------------------------------------------------
model = SB3_DQN(
    "MlpPolicy",
    env,
    verbose=0,
)

# ------------------------------------------------------------------
# 3. Entraînement
#    total_timesteps = nombre d'interactions (pas de jeu), pas d'épisodes.
# ------------------------------------------------------------------
TOTAL_TIMESTEPS = 50_000
model.learn(total_timesteps=TOTAL_TIMESTEPS)

# ------------------------------------------------------------------
# 4. Évaluation sur 100 épisodes
# ------------------------------------------------------------------
mean_reward, std_reward = evaluate_policy(model, env, n_eval_episodes=100)
print(f"\n🏆 Entraînement terminé ! ({TOTAL_TIMESTEPS} pas)")
print(f"📊 Récompense moyenne sur 100 épisodes : {mean_reward:.2f} +/- {std_reward:.2f}")

env.close()

c:\Formation\Projet_11\.venv\Lib\site-packages\stable_baselines3\common\evaluation.py:71: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(



🏆 Entraînement terminé ! (50000 pas)
📊 Récompense moyenne sur 100 épisodes : 25.51 +/- 143.83


In [ ]:
# ------------------------------------------------------------------
# 2. Instanciation du modèle
# ------------------------------------------------------------------
model = SB3_DQN(
    "MlpPolicy",
    env,
    verbose=0,
    learning_rate=6.3e-4,
    buffer_size=200000
)

# ------------------------------------------------------------------
# 3. Entraînement
#    total_timesteps = nombre d'interactions (pas de jeu), pas d'épisodes.
# ------------------------------------------------------------------
TOTAL_TIMESTEPS = 500_000
model.learn(total_timesteps=TOTAL_TIMESTEPS)

# ------------------------------------------------------------------
# 4. Évaluation sur 100 épisodes
# ------------------------------------------------------------------
mean_reward, std_reward = evaluate_policy(model, env, n_eval_episodes=100)
print(f"\n🏆 Entraînement terminé ! ({TOTAL_TIMESTEPS} pas)")
print(f"📊 Récompense moyenne sur 100 épisodes : {mean_reward:.2f} +/- {std_reward:.2f}")

env.close()


🏆 Entraînement terminé ! (500000 pas)
📊 Récompense moyenne sur 100 épisodes : 51.42 +/- 105.96


In [1]:
"""
Exemple : PPO sur un environnement à actions CONTINUES.

Rappel : DQN ne fonctionne que sur des action_space de type Discrete.
Ici, LunarLanderContinuous-v3 a un action_space de type Box(-1.0, 1.0, (2,))
-> 2 valeurs continues, chacune entre -1 et 1 :
   - action[0] : poussée du moteur principal (-1 = éteint, +1 = poussée max)
   - action[1] : moteur latéral (négatif = pousse à droite, positif = pousse à gauche)

PPO gère nativement les espaces continus : au lieu de prédire une Q-value
par action (impossible ici, il y a une infinité d'actions), le réseau
("acteur") prédit directement les paramètres d'une distribution de
probabilité continue (une gaussienne) sur les actions, dont on peut
ensuite échantillonner.
"""

import gymnasium as gym
import os
from stable_baselines3 import PPO
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.env_util import make_vec_env

ENV_NAME = "LunarLanderContinuous-v3"

# ------------------------------------------------------------------
# 1. Environnement(s)
#    PPO tire souvent parti d'environnements VECTORISÉS (plusieurs copies
#    de l'environnement jouées en parallèle) pour collecter des rollouts
#    plus rapidement -> make_vec_env crée n_envs instances indépendantes.
# ------------------------------------------------------------------
env = make_vec_env(ENV_NAME, n_envs=4)

# ------------------------------------------------------------------
# 2. Instanciation du modèle PPO
#    "MlpPolicy" fonctionne aussi bien en discret qu'en continu : SB3
#    adapte automatiquement la tête de sortie du réseau en lisant
#    env.action_space (Box -> tête gaussienne, Discrete -> tête softmax).
# ------------------------------------------------------------------
model = PPO(
    "MlpPolicy",
    env,
    verbose=0,
    gamma=0.999,
    ent_coef=0.01,
    tensorboard_log="./logs/",
)

# ------------------------------------------------------------------
# 3. Entraînement
# ------------------------------------------------------------------
TOTAL_TIMESTEPS = 1_000_000  # LunarLanderContinuous converge généralement autour de 300k-1M pas avec PPO
model.learn(total_timesteps=TOTAL_TIMESTEPS)

# ------------------------------------------------------------------
# 4. Évaluation
#    On évalue sur un environnement non-vectorisé, plus simple à lire.
# ------------------------------------------------------------------
eval_env = gym.make(ENV_NAME)
mean_reward, std_reward = evaluate_policy(model, eval_env, n_eval_episodes=100)
print(f"\n🏆 Entraînement terminé ! ({TOTAL_TIMESTEPS} pas)")
print(f"📊 Récompense moyenne sur 100 épisodes : {mean_reward:.2f} +/- {std_reward:.2f}")

# ------------------------------------------------------------------
# 5. Sauvegarde
# ------------------------------------------------------------------
os.makedirs(os.path.dirname("model/dqn_lunarlander"), exist_ok=True)
model.save("model/dqn_lunarlander")
print(f"\n💾 Modèle sauvegardé sous model/dqn_lunarlander.zip")
print("-> Placez ce fichier dans model/ avant de déployer le Space (voir README.md)")

eval_env.close()
env.close()


c:\Formation\Projet_11\.venv\Lib\site-packages\stable_baselines3\common\evaluation.py:71: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(



🏆 Entraînement terminé ! (1000000 pas)
📊 Récompense moyenne sur 100 épisodes : 261.51 +/- 35.63

💾 Modèle sauvegardé sous model/dqn_lunarlander.zip
-> Placez ce fichier dans model/ avant de déployer le Space (voir README.md)
